In [1]:
%pip install --upgrade --quiet pip
%pip install --upgrade --quiet optimum[openvino]
%pip install --upgrade --quiet langchain langchain-community langchain-core langchain-text-splitters
%pip install --upgrade --quiet faiss-cpu pypdf
%pip install --upgrade --quiet sentence-transformers hf_xet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [3]:
from huggingface_hub import login
import os

login()
os.environ['HUGGINGFACEHUB_API_TOKEN'] = '허깅페이스 토큰 넣으셈'

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

embedding_model_id = "BAAI/bge-m3"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_id,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("임베딩 모델 로드 완료!")


임베딩 모델 로드 완료!


In [ ]:
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import OpenVINOEmbeddings # OpenVINO 전용 임베딩
# from langchain_core.prompts import PromptTemplate

# # 1. 임베딩 모델 로드
# # 다국어(한국어 포함) 지원이 매우 뛰어나며, Dense(의미 검색), Sparse(키워드 검색), Multi-vector(정밀 검색) 방식을 모두 지원
# # 긴 문맥(8192 토큰)을 처리할 수 있어 RAG에 최적화
# # https://huggingface.co/BAAI/bge-m3
# embedding_model_id = "BAAI/bge-m3"


# embeddings = OpenVINOEmbeddings(
#     model_name_or_path=embedding_model_id, # 사용할 모델 ID
#     model_kwargs={"device": "GPU"}, # 실행 옵션=내장그래픽
#     encode_kwargs={"normalize_embeddings": True} # 벡터 간 유사도 계산(Cosine Similarity)을 정확히 하기 위해 벡터 값을 정규화
# )

# print("OpenVINO 임베딩 모델 로드 완료!")

In [5]:
%pip install openvino

Note: you may need to restart the kernel to use updated packages.


In [6]:
# 2. LLM 로드 (CPU환경에서 작동시키기 위해 OpenVINO 최적화)
from optimum.intel import OVModelForCausalLM
from transformers import AutoTokenizer, pipeline, StoppingCriteria, StoppingCriteriaList
from langchain_community.llms import HuggingFacePipeline
import torch

# 답변 생성용 LLM 로드
# 서울과기대에서 Llama 3를 기반으로 한국어 데이터를 대량으로 학습시킨 모델. 한국어 지식과 상식이 풍부함
# https://huggingface.co/MLP-KTLim/llama-3-Korean-Bllossom-8B
model_id = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

# Hugging Face에 있는 모델을 OpenVINO 포맷으로 변환하여 불러오기
ov_model = OVModelForCausalLM.from_pretrained(
    model_id,
    export=True, # 파이토치 모델을 OpenVINO 포맷으로 즉석에서 변환
    device="GPU",
    ov_config={"PERFORMANCE_HINT": "LATENCY"} # 모델을 응답 속도(Latency) 우선으로 최적화
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# 멈춤 신호 정의 클래스(지 혼자 안 멈추고 계속 질의응답하는 거 방지용)
class StopOnTokens(StoppingCriteria):
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # 모델이 생성한 마지막 토큰들이 아래 단어들과 일치하면 생성을 멈춤
        stop_ids = [
            tokenizer.convert_tokens_to_ids("<|eot_id|>"), # eos 토큰
            tokenizer.convert_tokens_to_ids("assistant"), # assistant라고 쓰려고 하면 멈춤
            tokenizer.convert_tokens_to_ids("user")       # user라고 쓰려고 하면 멈춤
        ]
        for stop_id in stop_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

# 파이프라인 생성
pipe = pipeline(
    "text-generation",
    model=ov_model,
    tokenizer=tokenizer,
    max_new_tokens=4096, # 최대 생성 가능한 답변 길이
    do_sample=True, # 확률에 따라 실행할 때마다 답변이 조금씩 달라지게 함
    temperature=0.1, # 정확한 답변을 위해 낮은 temperature와 높은 top p 설정
    top_p=0.9,
    repetition_penalty=1.0, # 반복 패널티
    return_full_text=False, # 프롬프트는 빼고 모델이 생성한 답변만 받음
    stopping_criteria=StoppingCriteriaList([StopOnTokens()]) # 중단 조건 적용
)

llm = HuggingFacePipeline(pipeline=pipe) # 생성된 파이프라인을 Langchain에서 사용할 수 있는 객체로 변환
print("LLM 로드 완료")

c:\Users\16Z90P\anaconda3\Lib\site-packages\openvino\runtime\__init__.py:10: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
c:\Users\16Z90P\anaconda3\Lib\site-packages\transformers\cache_utils.py:108: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.keys is None or self.keys.numel() == 0:
c:\Users\16Z90P\anaconda3\Lib\site-packages\transformers\masking_utils.py:190: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
c:\Users\16Z90P\anaconda3\Lib\site-packages\optimum\exporters\openvino\model_pat

NameError: name 'tensorflow' is not defined

In [ ]:
# 3. PDF 문서 로드
pdf_file_path = "수익기준서.pdf"

loader = PyPDFLoader(pdf_file_path) # PDF 파일을 읽어들이는 Loader를 생성
docs = loader.load() # 파일 내용을 읽어서 텍스트 데이터 리스트로 저장
print(f"문서 로드 완료: 총 {len(docs)} 페이지")

# 내용 미리보기
print(f"페이지 내용 일부:\n{docs[0].page_content[:200]}...")

문서 로드 완료: 총 517 페이지
페이지 내용 일부:
기업회계기준서 제1115호고객과의 계약에서 생기는 수익
한국회계기준원 회계기준위원회의결 2021.4.23....


In [ ]:
# 4. 텍스트 분할 (Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len
)

splits = text_splitter.split_documents(docs)
print(f"분할 완료: 총 {len(splits)}개의 Chunk 생성됨")
print(f"예시 Chunk:\n{splits[3].page_content[:200]}...")

분할 완료: 총 522개의 Chunk 생성됨
예시 Chunk:
이메일: webmaster@kasb.or.kr, 홈페이지: www.kasb.or.kr국제회계기준재단은 정부의 동의를 얻어 한국 내에서 사용하는 경우와 한국 내에 소재하는 기업의 해외 종속기업, 공동기업, 관계기업 또는 지점의 한국 이외 지역에서의 사용과 관련하여, 한국어로 구성된 일부 저작물에 대한 저작권을 주장할 권리를 포기했습니다. 이러한 저작물은 국제...


In [ ]:
# 5. 벡터 DB 생성
print("벡터 DB 생성 시작...")

# 위에서 로드한 OpenVINO embeddings 객체 사용
# Chunk들을 임베딩 모델을 사용해 벡터로 변환하고, FAISS라는 고속 검색 엔진에 저장
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

print("벡터 DB 생성 완료!")

벡터 DB 생성 시작...
벡터 DB 생성 완료!


In [ ]:
# 6. RAG 체인 구성
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Retriever 설정(벡터 DB를 검색기로 변환)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# 프롬프트 템플릿
llama_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 K-IFRS 회계 기준 전문가입니다. 다음 [참조 문서]를 바탕으로 사용자의 질문에 대해 명확하고 정확하게 답변해주세요.

[답변 작성 규칙]
1. 질문에 대한 핵심 내용만 간결하게 요약해서 답변하세요.
2. 불필요한 배경 설명(예: 도입 배경, US GAAP 비교, 라이선스 세부 사례 등)은 제외하세요.
3. n단계 모형을 설명할 때는 1단계부터 n단계까지 번호를 매겨서 명확히 구분하세요.
4. 각 단계 설명은 1~2문장으로 짧게 요약하세요.
5. 문서를 꼼꼼히 확인하고 사실과 다른 내용은 지어내지 마세요.

다음은 Few-shot 예시입니다. 
question: 확신유형의 보증과 용역유형의 보증에 대해 설명해줘.
answer: 확신유형의 보증은 수행의무로 회계처리하지 않고 용역유형의 보증은 수행의무로 회계처리한다.

Let's think step by step<|eot_id|><|start_header_id|>user<|end_header_id|>

[참조 문서]
{context}

[질문]
{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate.from_template(llama_template)

# 검색된 문서를 하나의 텍스트로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 답변 정제 함수(모델이 생성한 답변에서 불필요한 특수 기호나 태그를 지우는 후처리 작업)
def parse_final_answer(text):
    # 특수 토큰 제거
    text = text.replace("<|eot_id|>", "").replace("<|start_header_id|>", "").replace("<|end_header_id|>", "")
    
    # 모델이 답변 끝내고 혼자서 assistant 태그를 달고 또 말하는 것 방지
    keyword = "assistant"
    if keyword in text:
        text = text.split(keyword)[0]
    
    # user가 나와도 똑같이 잘라냄
    if "user" in text:
        text = text.split("user")[0]

    return text.strip()

# 체인 연결(전체 흐름을 연결하는 파이프라인)
rag_chain = (
    # retriever: 질문 관련 문서 검색
    # format_docs: 검색된 문서를 하나의 텍스트로 합침
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt # 프롬프트 템플릿에 질문과 문서를 삽입하는 역할
    | llm # 답변 생성 역할
    | StrOutputParser()
    | RunnableLambda(parse_final_answer) # 답변 다듬기
)

print("RAG 체인 구성 완료.")

RAG 체인 구성 완료.


In [ ]:
# 7. 질문 테스트
question = "수익인식의 5단계에 대해 설명해줘"

print(f"질문: {question}\n")
print("답변 생성 중...")

# 체인 실행
response = rag_chain.invoke(question)

print("="*50)
print(response)
print("="*50)

질문: 수익인식의 5단계에 대해 설명해줘

답변 생성 중...
수익인식의 5단계는 다음과 같습니다:

1. **고객과의 계약 식별**: 계약은 둘 이상의 당사자 사이에 집행 가능한 권리와 의무가 발생하게 하는 합의입니다. IFRS 15는 정해진 조건을 충족하는 고객과의 계약에만 적용됩니다. 때로는 여러 계약을 결합하여 하나의 계약으로 회계처리할 수 있습니다.

2. **수행의무 식별**: 하나의 계약은 고객에게 재화나 용역을 이전하는 여러 약속을 포함합니다. 이러한 재화나 용역이 구별될 수 있다면 약속은 수행의무로 별도로 회계처리됩니다. 고객이 재화나 용역 그 자체에서나 쉽게 구할 수 있는 다른 자원과 함께 효익을 얻을 수 있고, 약속을 계약 내의 다른 약속과 별도로 식별할 수 있다면 재화나 용역은 구별됩니다.

3. **거래가격 산정**: 거래가격은 고객에게 약속한 재화나 용역을 이전하고 그 대가로 기업이 받을 권리를 갖게 될 것으로 예상하는 금액입니다. 거래가격은 고정된 금액일 수도 있지만, 변동대가를 포함하거나 현금 외의 형태로 지급될 수도 있습니다. 거래가격은 계약에 포함된 유의적인 금융 요소가 있다면 화폐의 시간 가치 영향을 조정하며, 고객에게 지급하는 대가가 있는 경우에도 거래가격에서 조정됩니다. 변동대가는 변동대가와 관련된 불확실성이 나중에 해소될 때, 인식된 누적 수익 금액 중 유의적인 부분을 되돌리지 않는 가능성이 매우 높은 정도까지만 거래가격에 포함됩니다.

4. **거래가격 배분**: 거래가격은 일반적으로 계약에서 약속한 각 구별되는 재화나 용역의 상대적 개별 판매가격을 기준으로 배분됩니다. 개별 판매가격을 관측할 수 있는 경우, 각 수행의무에 대한 거래가격을 배분합니다. 이는 IFRS 15의 라이선스에 대한 상세한 적용지침을 통해 기업이 라이선스를 고객에게 이전하는 시기와 따라서 수익을 인식할 수 있는 시기를 판단하는 데 도움을 줍니다. 라이선스에 대한 수익을 인식하는 방식이 과거에 실무적으로 다양했기 때문에, IFRS 15에 적용지침을 

In [ ]:
# 벡터 DB(FAISS 인덱스)를 저장
vectorstore.save_local("./my_vector_db")
print("✅ 벡터 DB 저장 완료!")
# OpenVINO로 변환된 모델과 토크나이저를 저장
ov_model.save_pretrained("./my_llm_model")
tokenizer.save_pretrained("./my_llm_model")
print("✅ LLM 모델 저장 완료!")

✅ 벡터 DB 저장 완료!
✅ LLM 모델 저장 완료!
